# Known Limitations

A running log of data-quality and methodology limitations discovered while working with this dataset — some surfaced during cleaning (`01_cleaning.ipynb`), some during analysis (`02_EDA.ipynb`). Kept separate from those notebooks so caveats accumulate in one place as more turn up, instead of being scattered across (or bloating) the notebooks where they were found.

Each entry names where the issue was found, what it affects, and how to interpret results in light of it — not necessarily a fix, since some of these (like stale scrape data) can't be fixed after the fact.

## Table of Contents
1. [Stale Ratings on Discontinued Fragrances Skew Brand Rankings](#1-stale-ratings-on-discontinued-fragrances-skew-brand-rankings)
2. [Scraping Cap Understates The Woods Collection's Rated-Fragrance Count](#2-scraping-cap-understates-the-woods-collections-rated-fragrance-count)

## 1. Stale Ratings on Discontinued Fragrances Skew Brand Rankings

**Found in:** `02_EDA.ipynb` → Top Brands by Average Rating (Bayesian Weighted Rating, Top 15 chart)

**The issue**

Spot-checking Nr. 15 in the Top 15 Brands by Weighted Rating chart (Aubusson) against the live parfumo.com site turned up a large mismatch:
- The live site currently lists **68** Aubusson fragrances; this dataset has only **6**.
- None of those 6 currently carry any ratings on the live site — they're all old, discontinued releases.
- One of them still carries **516 ratings** in this dataset, and that historical rating volume (`Rating_Count`) is large enough on its own to have pushed Aubusson into the Top 15.

**Why it matters**

The Bayesian Weighted Rating formula (see `02_EDA.ipynb`, Top Brands section) weights each brand's own average against the population mean using `v`, the brand's total historical rating count — the more `v`, the more the brand's own average is trusted over the population mean. That logic assumes `v` reflects a brand's *current* standing. Here it doesn't: the 516 ratings are leftover volume from a scrape-time snapshot of a fragrance that no longer has any active rating signal, on a brand whose current catalog (68 fragrances) is barely represented in the data (6 fragrances). So a small, largely-stale slice of a brand can still out-rank far more current, better-represented brands.

**How to interpret results in light of this**

- Brand-level rankings (Top N by Weighted Rating, or any `Rating_Count`-weighted brand metric) reflect the state of the data *at scrape time*, not the current market — treat them as historical snapshots, not live standings.
- A brand with a small number of fragrances in this dataset relative to its real-world catalog size (as seen here, and separately in the "20-fragrance-per-brand cap" scraping artifact noted in `01_cleaning.ipynb`) is a signal to discount its ranking, especially if that catalog also skews old/discontinued.
- This was **not corrected in the ranking itself**. Manually excluding or re-weighting Aubusson because it happened to be the one spot-checked would be cherry-picking — the same staleness risk applies to any brand in the dataset, checked or not. Documenting it here keeps the finding honest without pretending a single manual fix makes the ranking reliable.

## 2. Scraping Cap Understates The Woods Collection's Rated-Fragrance Count

**Found in:** `02_EDA.ipynb` → Top Brands by Average Rating (Bayesian Weighted Rating, Top 15 chart) and its interpretation

**The issue**

The Top 15 brands split cleanly into two groups by number of rated fragrances (`n`): eight brands at `n` ≤ 24, seven at `n` ≥ 76 — nothing in between. Checking each low-`n` brand's *total* catalog size (rated + unrated) in the dataset explains one of them: **The Woods Collection** sits at exactly 20 total fragrances (17 rated) — the same hard ceiling identified in `01_cleaning.ipynb`'s "Per-Brand Fragrance Count Is Capped by Scraping" finding, where the scraper only pulled the first page of a brand's listing. Across the full dataset, 960 of 1,451 brands land at exactly 20 fragrances, versus only 9 at 19 and 4 at 21 — a distribution that only makes sense as a pagination artifact, not a natural spread of catalog sizes.

This was confirmed directly: the live parfumo.com site currently lists **29** The Woods Collection fragrances — 9 more than the 20 captured here — matching exactly what the pagination cap predicts (first page grabbed, the rest silently dropped).

Two other low-`n` brands in the Top 15, **Argos** and **Pana Dora**, were checked against the same signature and *ruled out*, on two counts:
- Both sit at 19 total fragrances in the cleaned data. Only 9 of 1,451 brands land at exactly 19 (versus 960 at 20), so 19 is not the cap value — it's rare enough to be a genuine catalog size, not a truncation. Being one below the cap isn't evidence of hitting it; if anything it's evidence the brand's catalog wasn't truncated, since a scraper stopping at "page 1" would stop at 20, not 19.
- Checked against the **raw, pre-cleaning scrape** (`02_Parfumo_Perfumes.csv`) as well, in case a brand had hit 20 originally and then lost a row during `01_cleaning.ipynb`'s dedup/invalid-row drops: both brands were already at 19 in the raw data, untouched by any cleaning step. There's no capped-then-trimmed count hiding underneath — 19 was the count from the very first row of data.

**Why it matters**

The Bayesian Weighted Rating trusts a brand's own average more as its total rating volume (`v`) grows. For The Woods Collection, `v` reflects only whatever fraction of the catalog got scraped, not the brand's real total rating volume — so its WR could shift once its full catalog is counted, in either direction. There's no way to tell from this data alone: the fragrances cut off by the cap could rate just as well as the visible ones, or notably worse.

**How to interpret results in light of this**

- A brand at exactly 20 total fragrances in this dataset (like Aubusson, entry 1, and The Woods Collection here) should be treated as a lower bound on both catalog size and rating volume, per the original `01_cleaning.ipynb` finding — and here, unlike entry 1, it's confirmed live rather than just inferred from the dataset-wide pattern.
- A brand merely low in count but *not* at exactly 20 (e.g. Argos, Pana Dora, both at 19 in both the raw and cleaned data) shouldn't be lumped in with the capped brands without checking — the exact-20 spike is specific enough that "close to 20" is not the same signal as "at 20."
- As with entry 1, this was **not corrected in the ranking**. Re-weighting or excluding The Woods Collection because a cap happens to affect roughly two-thirds of all brands in the dataset would require auditing all of them, not just the one that happened to land in this Top-15 chart.